# Article models A-J
Run the existing main.py and display results in the browser.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
# Run this notebook from the repository folder.
CODE_DIR = Path.cwd()
MAIN = CODE_DIR / 'main.py'
RESULT_ROOT = CODE_DIR / 'jupyter_results'
RESULT_ROOT.mkdir(exist_ok=True)
print(CODE_DIR, MAIN.exists())

In [ ]:
SCENARIO='I'; P=20; N=500; M=N; SEED=1
D_MAX=10; LAMBDA_C=0.005; LAMBDA_N=0.649 if N==500 else 0.611
LR=0.001; LAYERS=3; HIDDEN_DIM=256; PATIENCE=20; VAL_SPLIT=0.30; STOP_THRESHOLD=0.30; MAX_EPOCHS=800

In [ ]:
def run_model(model):
    out=RESULT_ROOT/f'{model}_{SCENARIO}_p{P}_n{N}_seed{SEED}'
    out.mkdir(parents=True, exist_ok=True)
    cmd=[sys.executable,'-X','utf8',str(MAIN),'--model',model,'--scenario',SCENARIO,'--p',str(P),'--n',str(N),'--m',str(M),'--h','0','--d_max',str(D_MAX),'--lambda_c',str(LAMBDA_C),'--lambda_n',str(LAMBDA_N),'--lr',str(LR),'--L',str(LAYERS),'--hidden_dim',str(HIDDEN_DIM),'--patience',str(PATIENCE),'--val_split',str(VAL_SPLIT),'--stop_threshold',str(STOP_THRESHOLD),'--max_epochs',str(MAX_EPOCHS),'--seed_start',str(SEED),'--n_seeds','1','--output_dir',str(out),'--save_ghat']
    env=dict(__import__('os').environ, PYTHONIOENCODING='utf-8', PYTHONUTF8='1')
    p=subprocess.run(cmd,cwd=CODE_DIR,text=True,capture_output=True,encoding='utf-8',errors='replace',env=env)
    print(p.stdout); print(p.stderr)
    f=out/f'results_seed{SEED}.json'
    return json.loads(f.read_text(encoding='utf-8')) if f.exists() else {'status':'failed','return_code':p.returncode}

In [ ]:
MODEL='A'
result=run_model(MODEL)
pd.Series(result)

In [ ]:
rows=[]
for model in 'ABCDEFGHIJ':
    print(f'===== Model {model} =====')
    r=run_model(model)
    rows.append({'model':model,'d_hat':r.get('d_hat',r.get('d_estimated')),'test_mse':r.get('test_mse'),'runtime_seconds':r.get('runtime_seconds'),'status':r.get('status')})
pd.DataFrame(rows)